# 02 — Baseline : le modèle avant fine-tuning

**Objectif :** mesurer la performance du modèle de base, **sans aucun entraînement**, sur le jeu de test. C'est le point de référence : le fine-tuning LoRA (notebook 03) ne sera utile que s'il fait nettement mieux.

**Modèle choisi : `Qwen2.5-Coder-1.5B-Instruct`**
- 1,5 milliard de paramètres : assez petit pour un GPU T4 gratuit, et réaliste pour un déploiement interne
- spécialisé en code, donc il connaît déjà la syntaxe SQL
- licence Apache 2.0 : utilisable en entreprise

**Métriques :**

| Métrique | Signification |
|---|---|
| **Execution accuracy** (principale) | La requête générée renvoie le même résultat que la requête de référence |
| Taux de SQL valide | La requête s'exécute sans erreur (même si le résultat est faux) |
| Exact match | Le texte SQL est identique à la référence (métrique stricte, donnée à titre indicatif) |


## 0. Installation

In [ ]:
import os, sys

REPO_URL = "https://github.com/eya-bouhmida/finance-text2sql-lora"
REPO_DIR = "/content/text2sql-finance-qlora"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    %cd {REPO_DIR}
    !pip install -q transformers accelerate peft datasets
    sys.path.append(REPO_DIR)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    sys.path.append("..")

DATA_DIR = "/content/drive/MyDrive/text2sql-finance-qlora/data" if IN_COLAB else "../data"
RESULTS_DIR = f"{REPO_DIR}/results" if IN_COLAB else "../results"

Already up to date.
/content/text2sql-finance-qlora


In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from src import data as D, model as M, evaluate as E

pd.set_option("display.max_colwidth", 150)
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN (active le GPU T4)")

## 1. Chargement du jeu de test

On utilise le fichier `test.jsonl` créé dans le notebook 01. Les **mêmes questions** seront utilisées pour évaluer le modèle fine-tuné dans le notebook 04, ce qui garantit une comparaison équitable.

In [ ]:
test_records = D.load_jsonl(f"{DATA_DIR}/test.jsonl")

# Pour aller plus vite pendant les essais, on peut limiter le nombre de questions.
# Mettre None pour évaluer tout le jeu de test (à faire pour les résultats finaux).
N_EVAL = None
if N_EVAL:
    import random
    random.Random(42).shuffle(test_records)
    test_records = test_records[:N_EVAL]

print(f"Questions évaluées : {len(test_records)}")
pd.Series([r["sql_complexity"] for r in test_records]).value_counts()

## 2. Chargement du modèle de base

In [ ]:
model, tokenizer = M.load_model(M.BASE_MODEL)

n_params = sum(p.numel() for p in model.parameters())
print(f"Modèle : {M.BASE_MODEL}")
print(f"Paramètres : {n_params / 1e9:.2f} milliards")
print(f"Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

### 2.1 Un premier essai

In [ ]:
ex = test_records[0]
for m in ex["messages"]:
    print(f"[{m['role'].upper()}]\n{m['content']}\n")

raw, _ = M.generate(model, tokenizer, [ex["messages"]])
print("=== Réponse brute du modèle ===\n", raw[0])
print("\n=== SQL extrait ===\n", E.extract_sql(raw[0]))
print("\n=== Requête de référence ===\n", ex["gold_sql"])
print("\n=== Évaluation ===\n", E.evaluate_one(ex["context"], ex["gold_sql"], E.extract_sql(raw[0])))

## 3. Génération sur tout le jeu de test

Décodage **glouton** (`do_sample=False`) : le modèle choisit toujours le token le plus probable, ce qui rend les résultats **reproductibles**.

In [ ]:
raw_outputs, sec_per_question = M.generate(
    model, tokenizer, [r["messages"] for r in test_records], batch_size=8
)
print(f"Temps moyen par question : {sec_per_question:.2f} s")

## 4. Résultats de la baseline

In [ ]:
results = E.evaluate_predictions(test_records, raw_outputs)
summary = E.summarize(results)
summary["sec_per_question"] = round(sec_per_question, 3)
summary["model"] = M.BASE_MODEL

pd.Series(summary)

In [ ]:
by_complexity = E.summarize_by(results, "sql_complexity")
by_complexity

In [ ]:
ax = by_complexity["execution_accuracy"].sort_values().plot.barh(figsize=(7, 4), color="#4C72B0")
ax.set_xlabel("Execution accuracy (%)")
ax.set_xlim(0, 100)
ax.set_title("Baseline : execution accuracy par complexité")
for i, v in enumerate(by_complexity["execution_accuracy"].sort_values()):
    ax.text(v + 1, i, f"{v:.0f} %", va="center")
plt.tight_layout(); plt.show()

## 5. Analyse des erreurs

On distingue deux types d'échecs :
- **SQL invalide** : la requête ne s'exécute pas (erreur de syntaxe, colonne inventée, fonction non SQLite…)
- **SQL valide mais faux** : la requête s'exécute mais ne renvoie pas le bon résultat (mauvaise jointure, mauvais filtre…)

In [ ]:
failures = results[~results["execution_match"]]
error_types = pd.Series({
    "SQL invalide": int((~failures["executable"]).sum()),
    "SQL valide mais résultat faux": int(failures["executable"].sum()),
})
print(error_types)

print("\nErreurs d'exécution les plus fréquentes :")
failures["pred_error"].dropna().str.split(":").str[0:2].str.join(":").value_counts().head(8)

In [ ]:
# Quelques exemples d'échecs, à commenter dans l'analyse
failures[["sql_complexity", "question", "gold_sql", "pred_sql", "pred_error"]].head(5)

## 6. Sauvegarde

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
results.to_csv(f"{RESULTS_DIR}/02_baseline_predictions.csv", index=False)
by_complexity.to_csv(f"{RESULTS_DIR}/02_baseline_by_complexity.csv")
E.save_json(summary, f"{RESULTS_DIR}/02_baseline_metrics.json")
print("Résultats sauvegardés dans", RESULTS_DIR)

## Conclusion de la baseline

*À compléter après exécution, par exemple :*

- Le modèle de base atteint **X %** d'execution accuracy sur les questions financières.
- **Y %** de ses requêtes ne s'exécutent même pas (principale cause : …).
- Il réussit bien les requêtes simples mais échoue sur … (jointures, sous-requêtes…).

**Prochaine étape :** notebook 03, fine-tuning avec LoRA pour corriger ces faiblesses.